In [1]:
import pandas as pd
import numpy as np
import re 
from pathlib import Path


In [2]:
df = pd.read_csv("../data/BMW/eng_ger_auto_reviews_dataset.csv", encoding="utf-8")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (21721, 12)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,Nikola,https://play-lh.googleusercontent.com/a-/ALV-U...,every time I tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,NaN,NaN,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,Itay Grudev,https://play-lh.googleusercontent.com/a-/ALV-U...,I have a vehicle with comfort access and a Sam...,2,0,5.11.4,2026-03-07 14:42:54,NaN,NaN,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,G. Żiogas,https://play-lh.googleusercontent.com/a-/ALV-U...,Can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,NaN,NaN,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,Jay Singh Josen,https://play-lh.googleusercontent.com/a/ACg8oc...,BMW stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,NaN,NaN,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,Paul Davies,https://play-lh.googleusercontent.com/a-/ALV-U...,The lack of support for Octopus Intelligent Go...,2,0,5.11.4,2026-03-06 19:38:16,NaN,NaN,5.11.4,BMW


In [3]:
#Dataset overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21721 entries, 0 to 21720
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              21721 non-null  object
 1   userName              21721 non-null  object
 2   userImage             21721 non-null  object
 3   content               21718 non-null  object
 4   score                 21721 non-null  int64 
 5   thumbsUpCount         21721 non-null  int64 
 6   reviewCreatedVersion  20492 non-null  object
 7   at                    21721 non-null  object
 8   replyContent          5372 non-null   object
 9   repliedAt             5372 non-null   object
 10  appVersion            20492 non-null  object
 11  company               21721 non-null  object
dtypes: int64(2), object(10)
memory usage: 2.0+ MB


### Remove Duplicate Reviews

Duplicate reviews were identified during the EDA phase.
We remove them based on the unique reviewId.

In [4]:
df = df.drop_duplicates(subset="reviewId").reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (11009, 12)


### Remove Missing Reviews

Reviews without textual content cannot be used for NLP analysis.

In [5]:
df = df.dropna(subset=["content"]).reset_index(drop=True)

In [6]:
#Check
df.isna().sum()

reviewId                   0
userName                   0
userImage                  0
content                    0
score                      0
thumbsUpCount              0
reviewCreatedVersion     579
at                         0
replyContent            8268
repliedAt               8268
appVersion               579
company                    0
dtype: int64

In [7]:
#Spalten prüfen - Verstehen welche Metadaten vorhanden sind
df.columns

Index(['reviewId', 'userName', 'userImage', 'content', 'score',
       'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
       'repliedAt', 'appVersion', 'company'],
      dtype='object')

### Remove Unnecessary Metadata Columns

Some metadata columns are not relevant for text analysis.

In [8]:
cols_to_drop = ["userName", "userImage", "replyContent", "repliedAt"]

df = df.drop(columns=cols_to_drop)

In [9]:
#Check
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time I tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,I have a vehicle with comfort access and a Sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,Can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,BMW stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,The lack of support for Octopus Intelligent Go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


##### Wir entfernen im nächsten Schritt html

In [10]:
import re 
def remove_html(text):
    return re.sub(r"<.*?>", "", text)
df["content"]=df["content"].apply(remove_html)
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time I tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,I have a vehicle with comfort access and a Sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,Can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,BMW stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,The lack of support for Octopus Intelligent Go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


### Normalize Text (Lowercase)

In [11]:
df["content"] = df["content"].str.lower()

### Normalize Whitespace

In [12]:
df["content"] = df["content"].str.replace(r"\s+", " ", regex=True).str.strip()

### Remove Very Short Reviews

In [13]:
df = df[df["content"].str.split().str.len() >= 2]

### Convert Timestamp

In [14]:
df["at"]=pd.to_datetime(df["at"], errors="coerce")

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10509 entries, 0 to 11007
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              10509 non-null  object        
 1   content               10509 non-null  object        
 2   score                 10509 non-null  int64         
 3   thumbsUpCount         10509 non-null  int64         
 4   reviewCreatedVersion  9953 non-null   object        
 5   at                    10509 non-null  datetime64[ns]
 6   appVersion            9953 non-null   object        
 7   company               10509 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 738.9+ KB


In [16]:
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


### Final Quality Check

In [17]:
print("Final dataset shape:", df.shape)

df.isna().sum()

Final dataset shape: (10509, 8)


reviewId                  0
content                   0
score                     0
thumbsUpCount             0
reviewCreatedVersion    556
at                        0
appVersion              556
company                   0
dtype: int64

### Save Clean Dataset

In [19]:
df.to_csv("../data/BMW/clean_reviews.csv", index=False)

##### Hinweis: Nach Projektinternem Austausch am 10.03.2026
Nach Diskussion mit Martin Lohr, haben wir uns dazu entschieden, keine Tokenization manuell vorzunehmen, da das SentenceTransformer-Modell (all-MiniLM-L6-v2) die tokenization intern übernimmt. Wenn wir manuell tokenisieren, würden wir die semantische Struktur zerstören und die Embeddings schlechter machen. Somit ist eine Textbereinigung mit HTML entfernen, lowercasing, Interpunktion löschen ausreichend.

## Preprocessing Summary

In diesem Notebook wurde eine strukturierte Datenbereinigung durchgeführt, um die Rohdaten für die nachfolgende Analyse mit Sentence Transformer Modellen vorzubereiten.

Der Preprocessing-Prozess begann mit dem Laden des ursprünglichen Datensatzes mit App-Reviews. Anschließend wurden mehrere Cleaning-Schritte durchgeführt, um die Datenqualität zu verbessern und potenzielle Störungen für spätere Analysen zu reduzieren.

### 1. Entfernung von Duplikaten
Zunächst wurden doppelte Reviews anhand der eindeutigen `reviewId` entfernt. Duplikate können die Analyse verzerren, da identische Inhalte mehrfach in statistische Auswertungen oder Embedding-Berechnungen eingehen würden.

### 2. Entfernung fehlender Inhalte
Reviews ohne Textinhalt (`content`) wurden gelöscht, da diese keinen semantischen Informationsgehalt besitzen und somit für NLP-Modelle nicht verwendbar sind.

### 3. Entfernen nicht benötigter Metadaten
Einige Spalten wie `userName`, `userImage`, `replyContent` und `repliedAt` wurden entfernt. Diese Informationen sind für die geplante Textanalyse und Embedding-Generierung nicht relevant und würden lediglich den Datensatz unnötig vergrößern.

### 4. Entfernen von HTML-Artefakten
HTML-Tags wurden aus den Review-Texten entfernt, da diese beim Web-Scraping entstehen können und keine semantische Bedeutung für die Textanalyse besitzen.

### 5. Normalisierung von Whitespace
Mehrfache Leerzeichen, Zeilenumbrüche und ähnliche Formatierungsartefakte wurden vereinheitlicht. Dies stellt sicher, dass die Texte in konsistenter Form vorliegen.

### 6. Entfernung sehr kurzer Reviews
Extrem kurze Reviews wurden entfernt, da sie häufig wenig semantische Information enthalten (z. B. „good“, „ok“ oder „nice“) und daher nur begrenzten analytischen Mehrwert bieten.

Nach diesen Schritten wurde der bereinigte Datensatz als `clean_reviews.csv` gespeichert.

---

## Gründe für bewusst ausgelassene Preprocessing-Schritte

Bestimmte klassische NLP-Preprocessing-Methoden wurden bewusst **nicht angewendet**, da moderne Transformer-Modelle mit natürlicher Sprache besser umgehen können, wenn der ursprüngliche Kontext erhalten bleibt.

### Stopword-Entfernung wurde ausgelassen
Stopwords wie „the“, „and“ oder „is“ können für Transformer-Modelle wichtige Kontextinformationen enthalten. Das Entfernen dieser Wörter kann den semantischen Zusammenhang eines Satzes verändern.

### Lemmatization und Stemming wurden nicht durchgeführt
Transformer-Modelle nutzen Subword-Tokenisierung und sind darauf trainiert, verschiedene Wortformen zu verstehen. Eine vorherige Normalisierung der Wortformen ist daher meist nicht notwendig und kann sogar Informationen verlieren.

### Tokenization wurde nicht manuell durchgeführt
Modelle wie Sentence Transformers besitzen eine integrierte Tokenisierung, die exakt auf das jeweilige Modell abgestimmt ist. Eine externe Tokenisierung wäre daher redundant.

### Punctuation wurde nicht entfernt
Satzzeichen können zusätzliche semantische Hinweise liefern, beispielsweise Emotionen oder Betonung (z. B. „!“ oder „?“). Da Transformer-Modelle mit solchen Informationen umgehen können, wurden diese bewusst beibehalten.

---

## Ergebnis

Das Ergebnis des Preprocessing-Schritts ist ein bereinigter Datensatz, der strukturelle Probleme entfernt, gleichzeitig aber den natürlichen Sprachkontext der Reviews bewahrt. Dadurch wird sichergestellt, dass die nachfolgenden Analysen und Embedding-Berechnungen mit Sentence Transformer Modellen auf qualitativ hochwertigen und semantisch vollständigen Textdaten basieren.